# SMX Quickstart: Binary Classification on a Synthetic Spectral Dataset

This notebook presents a complete and reproducible SMX workflow for binary classification in synthetic spectral data.

1. Define a controlled synthetic dataset with class-specific peak configurations
2. Partition data into calibration and test subsets
3. Apply mean-centering using calibration statistics only
4. Train an RBF-SVM classifier and derive probabilistic outputs
5. Fit the SMX explanation pipeline on calibration data
6. Rank predicates according to Local Reaching Centrality
7. Reconstruct top thresholds in spectral space
8. Perform complementary visual diagnostics (zone-wise and PCA-based)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
import smx
from smx import SMX, generate_synthetic_spectral_data
from smx.graph.interpretation import reconstruct_threshold_to_spectrum
import plotly.graph_objects as go
SEED = 42

The imported libraries define a reproducible computational environment for synthetic data generation, supervised classification, and post hoc explainability. The random seed is fixed to ensure deterministic behavior across executions.

## 1. Synthetic Spectral Dataset

The synthetic dataset is generated for two classes with partially overlapping Gaussian-like peak structures.

- **Class A**: peaks centered at 150, 300, and 500 (channel units)
- **Class B**: peaks centered at 150, 300, and 500 with distinct amplitudes and widths

The spectral axis is sampled at **300 points** from **1 to 600**, enabling controlled class separation while preserving realistic overlap.

In [ ]:
CLASSES_CONFIG = [
    {
        "name": "A",
        "n_samples": 116,
        "peaks": [
            # {'center' : 150, 'amplitude_mean' : 3.5, 'amplitude_std' : 0.5, 'width_mean' : 15.0, 'width_std' : 2.0},
            {'center' : 300, 'amplitude_mean' : 2.0, 'amplitude_std' : 0.3, 'width_mean' : 15.0, 'width_std' : 2.0},
            {'center' : 500, 'amplitude_mean' : 0.5, 'amplitude_std' : 0.3, 'width_mean' : 15.0, 'width_std' : 2.0},
        ],
        "noise_std": 0.4,
    },
    {
        "name": "B",
        "n_samples": 126,
        "peaks": [
            # {'center' : 150, 'amplitude_mean' : 4.3, 'amplitude_std' : 0.3, 'width_mean' : 17.0, 'width_std' : 2.0},
            {'center' : 300, 'amplitude_mean' : 0.8, 'amplitude_std' : 0.3, 'width_mean' : 14.0, 'width_std' : 1.5},
            {'center' : 500, 'amplitude_mean' : 0.45, 'amplitude_std' : 0.3, 'width_mean' : 15.0, 'width_std' : 2.0},
        ],
        "noise_std": 0.45,
    },
]

df = generate_synthetic_spectral_data(
    classes_config=CLASSES_CONFIG,
    n_points=300,
    x_min=1,
    x_max=600,
    seed=SEED,
)

X = df.drop(columns=["Class"])
y = df["Class"]

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{y.value_counts().to_string()}")
df.head()

This cell instantiates class-specific spectral signatures with controlled overlap in peak positions and class-dependent differences in peak intensity and width. The resulting structure is appropriate for evaluating whether SMX can recover discriminative spectral regions from a nontrivial, partially overlapping scenario.

In [ ]:
# Plotting the synthetic dataset with spectral cuts

spectral_cuts = [
    ('background 1', 1.0, 101.0),
    ('Feature 1', 101.0, 193.3),
    ('background 2', 193.3, 255.42),
    ('Feature 2', 255.42, 341.57),
    ('background 3', 341.57, 460.00),
    ('Feature 3', 460.756, 539.90),
    ('background 4', 539.90, 600.0)
]

data_complete = df.copy() # using the synthetic dataset generated above

class_col = "Class" if "Class" in data_complete.columns else data_complete.columns[0]
spec_cols = [c for c in data_complete.columns if c != class_col]
x_vars = np.array(spec_cols, dtype=float)

df_A = data_complete[data_complete[class_col] == "A"][spec_cols]
df_B = data_complete[data_complete[class_col] == "B"][spec_cols]

color_A = "#FA0F0F"
color_B = "#1365FC"

fig = go.Figure()

for i, (_, row) in enumerate(df_A.iterrows()):
    fig.add_trace(go.Scatter(
        x=x_vars,
        y=row.values,
        mode="lines",
        name="Class A",
        line=dict(color=color_A, width=1.0),
        legendgroup="A",
        showlegend=(i == 0),
    ))

for i, (_, row) in enumerate(df_B.iterrows()):
    fig.add_trace(go.Scatter(
        x=x_vars,
        y=row.values,
        mode="lines",
        name="Class B",
        line=dict(color=color_B, width=1.0),
        legendgroup="B",
        showlegend=(i == 0),
    ))

boundaries = sorted({cut[1] for cut in spectral_cuts} | {cut[2] for cut in spectral_cuts})
for b in boundaries:
    fig.add_vline(x=b, line=dict(color="gray", width=5, dash="dot"))

y_max = data_complete[spec_cols].values.max()
label_y = y_max * 1.02

for name, x_start, x_end in spectral_cuts:
    fig.add_annotation(
        x=(x_start + x_end) / 2,
        y=label_y,
        text=name,
        showarrow=False,
        textangle=-90,
        font=dict(size=20, color="black"),
        xanchor="center",
        yanchor="bottom",
    )

tick_max = int(np.ceil(x_vars.max() / 50.0) * 50)
ticks = np.arange(0, tick_max + 1, 50)

fig.update_layout(
    font=dict(size=19),
    xaxis_title="Variables (a.u.)",
    xaxis=dict(tickvals=ticks, ticktext=[str(v) for v in ticks]),
    yaxis_title="Intensity (a.u.)",
    yaxis=dict(range=[data_complete[spec_cols].values.min(), y_max * 1.35]),
    legend=dict(
        orientation="h",
        x=0.86,
        y=-0.3,
        xanchor="center",
        yanchor="bottom",
        font=dict(size=20),
    ),
    template="simple_white",
    #width=1250,
    #height=350,
)
fig.show()

The overlaid spectra and vertical boundaries provide a qualitative validation of the synthetic design. In particular, the visual segmentation into background and feature regions anticipates the subsequent zone-wise interpretation performed by SMX.

## 2. Calibration-Test Partition and Mean-Centering Preprocessing

The dataset is stratified to preserve class proportions across subsets. Mean-centering is then performed using calibration-set statistics only, preventing information leakage from the test set.

In [ ]:
X_cal, X_test, y_cal, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_cal = X_cal.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_cal = y_cal.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Mean-centre (fit on calibration set only)
X_mean = X_cal.mean()
X_cal_prep = X_cal - X_mean
X_test_prep = X_test - X_mean

print(f"Calibration set: {X_cal.shape}  |  Test set: {X_test.shape}")
print(f"Cal class distribution: {y_cal.value_counts().to_dict()}")
print(f"Test class distribution: {y_test.value_counts().to_dict()}")

The calibration-test split is stratified to preserve class balance. Mean-centering is estimated exclusively on calibration observations and then applied to both subsets, which ensures methodological rigor under a strict out-of-sample protocol.

## 3. SVM Classifier

An RBF-SVM is trained on preprocessed calibration data. Its probabilistic output is subsequently used as the continuous response variable required by the SMX perturbation analysis.

In [ ]:
svm = SVC(kernel="rbf", C=1.0, probability=True, random_state=SEED)
svm.fit(X_cal_prep, y_cal)

acc = (svm.predict(X_test_prep) == y_test).mean()
print(f"SVM test accuracy: {acc:.2%}")

# Continuous output used by SMX (probability of class A)
class_order = list(svm.classes_)
class_a_idx = class_order.index("A")
y_pred_cal = pd.Series(svm.predict_proba(X_cal_prep)[:, class_a_idx])
print(f"y_pred_cal range: [{y_pred_cal.min():.3f}, {y_pred_cal.max():.3f}]")

Model performance on the held-out test set quantifies predictive validity, whereas calibration-set class probabilities provide the continuous model response required by perturbation-based explainability in SMX.

## 4. SMX Pipeline

SMX is configured using the spectral partition defined in the notebook code:

- **background 1**: 1.0 to 101.0
- **Feature 1**: 101.0 to 193.3
- **background 2**: 193.3 to 255.42
- **Feature 2**: 255.42 to 341.57
- **background 3**: 341.57 to 460.00
- **Feature 3**: 460.756 to 539.90
- **background 4**: 539.90 to 600.0

This partition supports the interpretation of class-related discriminative regions while preserving non-informative baseline intervals.

In [ ]:
smx = SMX(
    spectral_cuts=spectral_cuts,
    quantiles=[ 0.2, 0.4, 0.6, 0.8],
    n_repetitions=4,
    n_bags=10,
    metric="perturbation",
    estimator=svm,
    perturbation_mode="median",
    perturbation_metric="probability_shift",
)

print("Fitting SMX pipeline…")
smx.fit(X_cal_prep, y_pred_cal, X_cal_natural=X_cal)
print("Done.")

At this stage, SMX is fitted on calibration spectra and corresponding model outputs. The selected hyperparameters control bagging stability, sampling fractions, and perturbation behavior, thereby defining the robustness and granularity of the explanatory graph.

## 5. Results: Top Predicates by Local Reaching Centrality

Predicates are ranked by Local Reaching Centrality (LRC), highlighting the most influential decision rules in the SMX explanatory graph.

In [ ]:
top = (
    smx.lrc_natural_[smx.lrc_natural_["Zone"].notna()]
    .sort_values("Local_Reaching_Centrality", ascending=False)
    [["Node", "Zone", "Operator", "Threshold_Natural", "Local_Reaching_Centrality"]]
    .reset_index(drop=True)
)

print("Top predicates by Local Reaching Centrality:")
top.head(30)

The ranked table identifies predicates with the strongest structural influence in the explanatory network. Higher Local Reaching Centrality values indicate conditions that propagate more broadly through downstream logical relations.

## 6. Threshold-to-Spectrum Reconstructions

For each selected predicate, the threshold in latent space is reconstructed into spectral space and superimposed on calibration spectra grouped by class. This representation links abstract model constraints to physically interpretable spectral profiles.

In [ ]:
CLASS_COLORS = {"A": "gold", "B": "blue"}

def plot_zone_inline(lrc_natural_df, row_index, spectral_zones_original,
                     pca_info_dict_original, y_labels, class_colors=None):
    """Reconstruct a threshold to spectrum space and display the plot inline."""
    if class_colors is None:
        class_colors = {"A": "gold", "B": "blue"}

    zone_name = lrc_natural_df.iloc[row_index]["Zone"]
    threshold_score = float(lrc_natural_df.iloc[row_index]["Threshold_Natural"])

    threshold_spectrum = reconstruct_threshold_to_spectrum(
        threshold_value=threshold_score,
        zone_name=zone_name,
        pca_info_dict=pca_info_dict_original,
    )

    zone_df = spectral_zones_original[zone_name]
    x_values = pd.to_numeric(zone_df.columns, errors="coerce")

    fig = go.Figure()
    seen_classes = set()
    for idx, row in zone_df.iterrows():
        class_label = y_labels.iloc[idx] if idx < len(y_labels) else "Unknown"
        show_legend = class_label not in seen_classes
        seen_classes.add(class_label)
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=row.values,
                mode="lines",
                line=dict(color=class_colors.get(class_label, "rgba(128,128,128,0.3)"), width=0.5),
                name=f"Class {class_label}",
                legendgroup=class_label,
                showlegend=show_legend,
                hoverinfo="skip",
            )
        )

    node_natural = lrc_natural_df.iloc[row_index].get("Node_Natural", "")
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=threshold_spectrum.values,
            mode="lines",
            line=dict(color="red", width=4, dash="dash"),
            name=f"Threshold Spectrum ({threshold_spectrum.name})",
        )
    )

    fig.update_layout(
        title=f"Zone '{zone_name}' — Multivariate Threshold (Predicate: {node_natural})",
        xaxis_title="Variables (Artificial Units)",
        yaxis_title="Intensity (Arbitrary Units)",
        template="plotly_white",
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )
    fig.show()

for _, row in top[:10].iterrows():
    zone_name = row["Zone"]
    row_index = smx.lrc_natural_.index[
        smx.lrc_natural_["Node"] == row["Node"]
    ].tolist()[0]
    plot_zone_inline(
        lrc_natural_df=smx.lrc_natural_,
        row_index=row_index,
        spectral_zones_original=smx.zones_natural_,
        pca_info_dict_original=smx.pca_info_natural_,
        y_labels=y_cal,
        class_colors=CLASS_COLORS,
    )

These reconstructions project latent thresholds back to observed-variable space, enabling direct inspection of how high-ranking predicates intersect empirical class spectra within each zone.

In [ ]:
top_per_zone = (
    smx.lrc_natural_[smx.lrc_natural_["Zone"].notna()]
    .sort_values("Local_Reaching_Centrality", ascending=False)
    [["Node", "Zone", "Operator", "Threshold_Natural", "Local_Reaching_Centrality"]].drop_duplicates(subset=["Zone"])
    .reset_index(drop=True)
)

print("Top predicates per zone by Local Reaching Centrality:")
display(top_per_zone)

Selecting one predicate per zone yields a compact interpretability profile that reduces redundancy while preserving spatial coverage across the full spectral domain.

## 7. Best Predicate per Spectral Zone

To avoid redundancy, only the highest-LRC predicate is retained for each zone, yielding a concise and zone-balanced interpretability summary.

In [ ]:
for _, row in top_per_zone[:5].iterrows():
    zone_name = row["Zone"]
    row_index = smx.lrc_natural_.index[
        smx.lrc_natural_["Node"] == row["Node"]
    ].tolist()[0]
    plot_zone_inline(
        lrc_natural_df=smx.lrc_natural_,
        row_index=row_index,
        spectral_zones_original=smx.zones_natural_,
        pca_info_dict_original=smx.pca_info_natural_,
        y_labels=y_cal,
        class_colors=CLASS_COLORS,
    )

The following diagnostic figures complement predicate-level interpretation by characterizing global class structure, intra-class variability, and zone-level latent representations.

In [ ]:
# Full calibration spectra with spectral-zone shading
import plotly.graph_objects as go
x_full = pd.to_numeric(X_cal_prep.columns, errors='coerce')
colors_by_class = {'A': 'gold', 'B': 'blue'}
legend_added = set()
fig = go.Figure()
for idx, row in X_cal_prep.iterrows():
    cls = y_cal.iloc[idx]
    show_legend = cls not in legend_added
    legend_added.add(cls)
    fig.add_trace(go.Scatter(
        x=x_full,
        y=row.values,
        mode='lines',
        line=dict(color=colors_by_class.get(cls, 'rgba(128,128,128,0.3)'), width=0.5),
        opacity=0.6,
        name=f'Class {cls}',
        legendgroup=cls,
        showlegend=show_legend,
        hoverinfo='skip'
    ))
# Add spectral zone background shading
zone_fill = ['rgba(180,180,220,0.2)', 'rgba(220,220,180,0.2)']
for i, (zname, zstart, zend) in enumerate(spectral_cuts):
    fig.add_vrect(x0=zstart, x1=zend, fillcolor=zone_fill[i % 2], layer='below', line_width=0.5, line_color='rgba(100,100,100,0.3)', annotation_text=zname, annotation_position='top', annotation=dict(font_size=8, textangle=-90))
fig.update_layout(title='Calibration spectra with spectral-zone shading', xaxis_title='Variables (Artificial Units)', yaxis_title='Intensity (a.u.)', template='plotly_white', height=520, legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99))
fig.show()

This visualization presents all calibration spectra with zone shading, facilitating visual inspection of how signal morphology differs between classes across predefined intervals.

In [ ]:
# Mean ± std envelope per class
import plotly.graph_objects as go
x_full = pd.to_numeric(X_cal_prep.columns, errors='coerce')
class_colors = {'A': 'gold', 'B': 'blue'}
fig = go.Figure()
for cls, color in class_colors.items():
    mask = (y_cal == cls)
    subset = X_cal_prep[mask.values].values
    mean_s = subset.mean(axis=0)
    std_s = subset.std(axis=0)
    fig.add_trace(go.Scatter(x=x_full, y=mean_s + std_s, mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=x_full, y=mean_s - std_s, mode='lines', line=dict(width=0), fill='tonexty', fillcolor=color.replace('gold', 'rgba(255,215,0,0.15)').replace('blue', 'rgba(0,0,255,0.15)'), showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=x_full, y=mean_s, mode='lines', line=dict(color=color, width=2), name=f'Class {cls} mean'))
for i, (zname, zstart, zend) in enumerate(spectral_cuts):
    fig.add_vrect(x0=zstart, x1=zend, fillcolor=['rgba(180,180,220,0.15)', 'rgba(220,220,180,0.15)'][i % 2], layer='below', line_width=0)
fig.update_layout(title='Mean ± Std per Class (calibration set)', xaxis_title='Variables (Artificial Units)', yaxis_title='Intensity (a.u.)', template='plotly_white', height=500, legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99))
fig.show()

Class-specific mean and standard-deviation envelopes summarize central tendency and dispersion, clarifying where between-class separation exceeds within-class variability.

In [ ]:
# PCA aggregation per spectral zone (PC1 scores) and violin plot
# Extract spectral zones from preprocessed calibration set
from smx import extract_spectral_zones, ZoneAggregator
spectral_zones_cal = extract_spectral_zones(X_cal_prep, spectral_cuts)
agg = ZoneAggregator(method='pca')
agg.fit(spectral_zones_cal)
zone_scores_df = agg.transform(spectral_zones_cal)
pca_info_dict = agg.pca_info_
zone_cols = zone_scores_df.columns.tolist()

# Split violin: PC1 scores per spectral zone by class
import plotly.graph_objects as go

zone_cols = zone_scores_df.columns.tolist()
class_colors = {'A': 'rgba(255, 0, 0, 1.0)', 'B': 'rgba(0, 102, 255, 1.0)'}
line_colors = {'A': "#5E0404", 'B': "#000766"}
class_sides = {'A': 'negative', 'B': 'positive'}
class_names = {'A': 'Class A', 'B': 'Class B'}
fig = go.Figure()

for cls, color in class_colors.items():
    mask = (y_cal == cls).values
    for zone in zone_cols:
        fig.add_trace(go.Violin(
            x=[zone] * mask.sum(),
            y=zone_scores_df.loc[mask, zone].values,
            name=class_names[cls],
            legendgroup=cls,
            scalegroup=cls,
            showlegend=(zone == zone_cols[0]),
            side=class_sides[cls],
            line_color=line_colors[cls],
            fillcolor=color,
            opacity=1.0,
            box_visible=False,
            meanline_visible=True,
            points=False,
            width=0.6
        ))

fig.update_layout(
    font=dict(size=17),
    title=dict(
        text="Scores by Spectral Zone and Class",
        x=0.5,
        xanchor='center',
        y=0.88,
    ),
    xaxis_title='Spectral Zone',
    yaxis_title='PC 1 scores',
    template='simple_white',
    height=700,
    width=1300,
    xaxis=dict(tickangle=-30),
    legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.95, font=dict(size=16))
)
fig.show()


The split-violin representation provides a compact comparative view of class-specific score distributions in each zone, emphasizing directionality and magnitude of separation. Together with LRC ranking, these diagnostics support a coherent interpretation of predictive mechanisms across spectral regions.

In [ ]:
# PC1 explained variance per spectral zone (bar plot)
import plotly.graph_objects as go
zones_var = [(z, pca_info_dict[z]['variance_explained']) for z in pca_info_dict]
zones_var.sort(key=lambda x: x[1], reverse=True)
zone_names_sorted, var_expl = zip(*zones_var)
fig = go.Figure(go.Bar(x=list(zone_names_sorted), y=[v * 100 for v in var_expl], marker_color=[f'rgba(31,119,180,{0.4 + 0.6 * v})' for v in var_expl], text=[f'{v*100:.1f}%' for v in var_expl], textposition='outside'))
fig.update_layout(title='PC1 Explained Variance per Spectral Zone', xaxis_title='Spectral Zone', yaxis_title='Variance Explained by PC1 (%)', xaxis=dict(tickangle=-45), yaxis=dict(range=[0, 105]), template='plotly_white', height=480)
fig.show()

The explained-variance profile indicates how efficiently each spectral zone is summarized by its first principal component, which informs the reliability of one-dimensional zone representations.

## 8. Grouped Background Zones

A new feature in SMX allows multiple spectral cuts to be **merged into a single zone** by assigning them the same `group` value.  Instead of treating each background interval independently, all background regions are pooled into one zone called `"background"`.  The same 3-element tuple format from earlier still works; only the background cuts receive a 4th element.

This impacts both the PCA aggregation step (PC1 is now computed over the full concatenated background) and the perturbation metric (all background columns are zeroed simultaneously).  Feature zones remain unchanged.

In [ ]:
# Spectral-cut definition with grouped background zones.
# Background intervals share group='background'; feature cuts remain as plain 3-tuples.
spectral_cuts_grouped = [
    ('background 1', 1.0,     101.0,  'background'),
    ('Feature 1',    101.0,   193.3),
    ('background 2', 193.3,   255.42, 'background'),
    ('Feature 2',    255.42,  341.57),
    ('background 3', 341.57,  460.00, 'background'),
    ('Feature 3',    460.756, 539.90),
    ('background 4', 539.90,  600.0,  'background'),
]

smx_grouped = SMX(
    spectral_cuts=spectral_cuts_grouped,
    quantiles=[0.2, 0.4, 0.6, 0.8],
    n_repetitions=4,
    n_bags=10,
    metric="perturbation",
    estimator=svm,
    perturbation_mode="median",
    perturbation_metric="probability_shift",
)

print("Fitting SMX pipeline with grouped background zones…")
smx_grouped.fit(X_cal_prep, y_pred_cal, X_cal_natural=X_cal)
print("Done.")
print(f"\nZones: {list(smx_grouped.zone_scores_.columns)}")

In [ ]:
top_grouped = (
    smx_grouped.lrc_natural_[smx_grouped.lrc_natural_["Zone"].notna()]
    .sort_values("Local_Reaching_Centrality", ascending=False)
    [["Node", "Zone", "Operator", "Threshold_Natural", "Local_Reaching_Centrality"]]
    .reset_index(drop=True)
)

print("Top predicates by LRC (grouped background):")
top_grouped.head(30)

In [ ]:
# Split-violin of PC1 scores per zone — grouped version
# The four separate background zones are now a single "background" zone.
zone_scores_grouped_df = smx_grouped.zone_scores_.copy()
zone_cols_grouped = zone_scores_grouped_df.columns.tolist()

class_colors = {'A': 'rgba(255, 0, 0, 1.0)', 'B': 'rgba(0, 102, 255, 1.0)'}
line_colors  = {'A': "#5E0404",               'B': "#000766"}
class_sides  = {'A': 'negative',              'B': 'positive'}
class_names  = {'A': 'Class A',               'B': 'Class B'}

fig = go.Figure()
for cls, color in class_colors.items():
    mask = (y_cal == cls).values
    for zone in zone_cols_grouped:
        fig.add_trace(go.Violin(
            x=[zone] * mask.sum(),
            y=zone_scores_grouped_df.loc[mask, zone].values,
            name=class_names[cls],
            legendgroup=cls,
            scalegroup=cls,
            showlegend=(zone == zone_cols_grouped[0]),
            side=class_sides[cls],
            line_color=line_colors[cls],
            fillcolor=color,
            opacity=1.0,
            box_visible=False,
            meanline_visible=True,
            points=False,
            width=0.6,
        ))

fig.update_layout(
    font=dict(size=17),
    title=dict(
        text="PC1 Scores by Spectral Zone and Class — Grouped Background",
        x=0.5, xanchor='center', y=0.88,
    ),
    xaxis_title='Spectral Zone',
    yaxis_title='PC 1 scores',
    template='simple_white',
    height=700,
    width=1300,
    xaxis=dict(tickangle=-30),
    legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.95, font=dict(size=16)),
)
fig.show()

In [ ]:

# ── Diagnostic comparison: ungrouped vs grouped backgrounds ──────────────────
# Re-run the ungrouped SMX so both objects are in scope
from smx import SMX

smx_ungrouped = SMX(
    spectral_cuts=spectral_cuts,
    quantiles=[0.2, 0.4, 0.6, 0.8],
    n_repetitions=4,
    n_bags=10,
    metric="perturbation",
    estimator=svm,
    perturbation_mode="median",
    perturbation_metric="probability_shift",
)
smx_ungrouped.fit(X_cal_prep, y_pred_cal, X_cal_natural=X_cal)

# ── 1. Zone inventory ────────────────────────────────────────────────────────
print("=== 1. Zone inventory ===")
print(f"  Ungrouped zones ({len(smx_ungrouped.zone_scores_.columns)}): {list(smx_ungrouped.zone_scores_.columns)}")
print(f"  Grouped zones   ({len(smx_grouped.zone_scores_.columns)}): {list(smx_grouped.zone_scores_.columns)}")

# ── 2. PC1 variance explained per zone ──────────────────────────────────────
print("\n=== 2. PC1 variance explained ===")
print(f"  {'Zone':<18}  {'Ungrouped':>10}  {'Grouped':>10}")
all_zones = sorted(set(list(smx_ungrouped.pca_info_.keys()) + list(smx_grouped.pca_info_.keys())))
for z in all_zones:
    ug = smx_ungrouped.pca_info_.get(z, {}).get("variance_explained")
    gr = smx_grouped.pca_info_.get(z, {}).get("variance_explained")
    ug_str = f"{ug:.3f}" if ug is not None else "—"
    gr_str = f"{gr:.3f}" if gr is not None else "—"
    print(f"  {z:<18}  {ug_str:>10}  {gr_str:>10}")

# ── 3. LRC rank of background zones ─────────────────────────────────────────
def zone_lrc_stats(lrc_df, zone_name):
    sub = lrc_df[lrc_df["Zone"] == zone_name]["Local_Reaching_Centrality"]
    if sub.empty:
        return None, None, None
    return sub.max(), sub.mean(), len(sub)

print("\n=== 3. LRC of background zones ===")
bg_zones_ug = [z for z in smx_ungrouped.zone_scores_.columns if "background" in z.lower()]
for z in bg_zones_ug:
    mx, mn, n = zone_lrc_stats(smx_ungrouped.lrc_natural_, z)
    print(f"  Ungrouped '{z}': max_LRC={mx:.4f}, mean_LRC={mn:.4f}, n_predicates={n}")

mx, mn, n = zone_lrc_stats(smx_grouped.lrc_natural_, "background")
print(f"  Grouped  'background': max_LRC={mx:.4f}, mean_LRC={mn:.4f}, n_predicates={n}")

# ── 4. Top-5 per-zone LRC: ungrouped vs grouped ──────────────────────────────
print("\n=== 4. Top predicate per zone (ungrouped) ===")
top1 = (smx_ungrouped.lrc_natural_.dropna(subset=["Zone"])
        .sort_values("Local_Reaching_Centrality", ascending=False)
        .drop_duplicates(subset=["Zone"])
        [["Zone", "Operator", "Threshold_Natural", "Local_Reaching_Centrality"]])
print(top1.to_string(index=False))

print("\n=== 5. Top predicate per zone (grouped) ===")
top2 = (smx_grouped.lrc_natural_.dropna(subset=["Zone"])
        .sort_values("Local_Reaching_Centrality", ascending=False)
        .drop_duplicates(subset=["Zone"])
        [["Zone", "Operator", "Threshold_Natural", "Local_Reaching_Centrality"]])
print(top2.to_string(index=False))
